### Day 2 Lab Assignment\n- Multi-Modal Nutrition AI Agent\n- tools orchestration\n- context engineering (trim messages + summarization memory)\n- Streamlit UI integration

In [ ]:
system_prompt = \"\"\"\nYou are a Nutrition AI Assistant Agent.\nAnalyze meal text and optional food images.\nDynamically use tools only when needed:\n- search_healthy_options for healthy places/nutrition info\n- store_meal_record when user asks to store data\nDo not provide medical diagnosis or strict diet plans.\nAlways include disclaimer:\nThis is not medical or dietary advice. Consult a qualified professional.\n\"\"\"

In [ ]:
from tools import search_healthy_options, store_meal_record\nfrom memory import trim_messages, SummaryMemoryManager\nfrom langchain.agents import create_agent\nfrom langgraph.checkpoint.memory import InMemorySaver

In [ ]:
nutrition_agent = create_agent(\n    \"gpt-4.1\",\n    system_prompt=system_prompt,\n    tools=[search_healthy_options, store_meal_record],\n    checkpointer=InMemorySaver(),\n    middleware=[trim_messages],\n)\nnutrition_agent

### Basic meal analysis example

In [ ]:
from langchain.messages import HumanMessage\nres = nutrition_agent.invoke(\n    {\"messages\": [HumanMessage(content=\"I ate grilled chicken, rice, and salad for dinner. Analyze it.\")]},\n    config={\"configurable\": {\"thread_id\": \"nutrition-demo-1\"}},\n)\nprint(res[\"messages\"][-1].content)

### Multi-modal example (text + image)

In [ ]:
import base64\nfrom pathlib import Path\n\n# Put your image in Day2/lab/sample_meal.jpg\nimage_path = Path(\"sample_meal.jpg\")\nif image_path.exists():\n    encoded = base64.b64encode(image_path.read_bytes()).decode(\"utf-8\")\n    multimodal_res = nutrition_agent.invoke(\n        {\n            \"messages\": [\n                HumanMessage(content=[\n                    {\"type\": \"text\", \"text\": \"Analyze this meal photo and estimate nutrition.\"},\n                    {\"type\": \"image\", \"base64\": encoded, \"mime_type\": \"image/jpeg\"},\n                ])\n            ]\n        },\n        config={\"configurable\": {\"thread_id\": \"nutrition-demo-2\"}},\n    )\n    print(multimodal_res[\"messages\"][-1].content)\nelse:\n    print(\"sample_meal.jpg not found - add one to test multi-modal input.\")

### Context optimization example (summarization memory)

In [ ]:
memory_manager = SummaryMemoryManager(model_name=\"gpt-4.1-mini\", max_messages=4)\nmessages = [\n    HumanMessage(content=\"My goal is fat loss with enough protein.\"),\n    HumanMessage(content=\"I usually eat late at night.\"),\n    HumanMessage(content=\"Today breakfast was eggs and toast.\"),\n    HumanMessage(content=\"Lunch was koshari and juice.\"),\n    HumanMessage(content=\"Dinner was shawarma sandwich.\"),\n]\noptimized = memory_manager.inject_summary(messages)\nprint(f\"Original count: {len(messages)}\")\nprint(f\"Optimized count: {len(optimized)}\")\nprint(optimized[0].content if optimized else \"No summary\")

### Streamlit app\nRun this from terminal:\n`streamlit run app.py`